In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt


In [2]:
 torch.manual_seed(42)

In [4]:
df = pd.read_csv("fmnist_small.csv")
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,9,0,0,0,0,0,0,0,0,0,...,0,7,0,50,205,196,213,165,0,0
1,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,1,0,0,0,...,142,142,142,21,0,3,0,0,0,0
3,8,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,8,0,0,0,0,0,0,0,0,0,...,213,203,174,151,188,10,0,0,0,0


In [5]:
x = df.iloc[:,1:].values
y = df.iloc[:,0].values

In [6]:
X_train , X_test , y_train, y_test = train_test_split(x,y,test_size=0.2,random_state=42)

In [9]:
X_train.shape

(4800, 784)

In [8]:
x_train = X_train/255.0
x_test = X_test/255.0

In [43]:
class CustomerDataset(Dataset):

  def __init__(self,features,labels):

    self.features = torch.tensor(features,dtype=torch.float32)
    self.labels = torch.tensor(labels,dtype=torch.long)

  def __len__(self):
    return len(self.features)

  def __getitem__(self,idx):
      return self.features[idx],self.labels[idx]

In [44]:
train_dataset = CustomerDataset(x_train,y_train)

In [45]:
len(train_dataset)

4800

In [46]:
test_dataset = CustomerDataset(x_test,y_test)

In [47]:
len(test_dataset)

1200

In [48]:
train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader = DataLoader(test_dataset,batch_size=32,shuffle=False)

In [54]:
class MyNN(nn.Module):

  def __init__(self,num_features):

    super().__init__()
    self.model = nn.Sequential(
        nn.Linear(num_features,128),
        nn.ReLU(),
        nn.Linear(128,64),
        nn.ReLU(),
        nn.Linear(64,10)
    )

  def forward(self,x):
    return self.model(x)

In [55]:
epochs = 100
learning_rate  = 0.1

In [58]:
model = MyNN(X_train.shape[1])

criterion = nn.CrossEntropyLoss()

optimizer = optim.SGD(model.parameters(),lr=learning_rate)

In [59]:
for epoch in range(epochs):
  total_epoch_loss = 0
  for batch_features ,batch_labels in train_loader:

    output = model(batch_features)

    loss = criterion(output,batch_labels)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    total_epoch_loss += loss.item()

  avg_loss = total_epoch_loss/len(train_loader)
  print(f"Epoch {epoch+1}/{epochs} Loss: {avg_loss}")

Epoch 1/100 Loss: 1.3183868722120922
Epoch 2/100 Loss: 0.7700774955749512
Epoch 3/100 Loss: 0.6688790921370188
Epoch 4/100 Loss: 0.5930577810605367
Epoch 5/100 Loss: 0.5432835607727369
Epoch 6/100 Loss: 0.4961409282684326
Epoch 7/100 Loss: 0.4743771757682165
Epoch 8/100 Loss: 0.4489046417673429
Epoch 9/100 Loss: 0.42531372209390006
Epoch 10/100 Loss: 0.39926519493261975
Epoch 11/100 Loss: 0.37661464189489685
Epoch 12/100 Loss: 0.37382458984851835
Epoch 13/100 Loss: 0.3570463068286578
Epoch 14/100 Loss: 0.33525810346007345
Epoch 15/100 Loss: 0.33631339167555174
Epoch 16/100 Loss: 0.31411220903197923
Epoch 17/100 Loss: 0.30112324809034663
Epoch 18/100 Loss: 0.2904119360198577
Epoch 19/100 Loss: 0.2743731716275215
Epoch 20/100 Loss: 0.2729728526622057
Epoch 21/100 Loss: 0.2519098488986492
Epoch 22/100 Loss: 0.24259115380545457
Epoch 23/100 Loss: 0.24407134249806403
Epoch 24/100 Loss: 0.22662204784651596
Epoch 25/100 Loss: 0.22772288056711357
Epoch 26/100 Loss: 0.20998145149399836
Epoch 27

In [60]:
model.eval()

MyNN(
  (model): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [63]:
total = 0
correct = 0

with torch.no_grad():

  for batch_features ,batch_labels in test_loader:

    output = model(batch_features)

    _, predictions = torch.max(output, 1) # Get predictions (indices of max values) instead of using built-in max()

    total += batch_labels.size(0) # Increment total samples by batch size

    correct += (predictions == batch_labels).sum().item() # Accumulate correct predictions

print(f"Accuracy: {correct / total * 100:.2f}%")

Accuracy: 83.58%
